# Opis

Krótki notebook, który pozwala przetestować działanie różnych elementów implementacyjnych w szybki sposób.

# Importy

In [1]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import tqdm
import wandb
import json
sys.path.append('../') # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset

W0604 23:20:24.226000 3164 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Przetwarzanie

## Pomocnicza klasa tworząca sztuczne przykłady osobników

In [2]:
class FramsticksDummyDataset(Dataset):
    """
    Tworzy zbiór losowych grafów o stałym rozmiarze, symulujących dane z Framsticks.
    Do weryfikacji czy model działa, uczy się i nie pojawiają się błędy.
    """
    def __init__(self, num_samples=1000, max_nodes=15, in_channels=3):
        self.num_samples = num_samples
        self.max_nodes = max_nodes
        self.in_channels = in_channels

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
		# Tworzy macierz max_nodes x in_channels z wartościami w zakresie -1 do 1
        x = torch.rand((self.max_nodes, self.in_channels)) * 2 - 1.0

        # Generuje losową, symetryczną macierz sąsiedztwa A z zerami i jedynkami
        adj = torch.rand((self.max_nodes, self.max_nodes))
		# Sztuczka, aby zapewnić symetrię
        adj = (adj + adj.T) / 2
		# Progowanie na 0 i 1 korzystając z odcięcia (większe odcięcie, rzadsza struktura
        adj = (adj > 0.7).float()
		# Wypełnienie diagonali
        adj.fill_diagonal_(1.0)

        return x, adj




## Załadowanie danych i przygotowanie do przetwarzania

In [10]:
# dataset = FramsticksDummyDataset(num_samples=1000)
torch.set_float32_matmul_precision('medium')
genotypes = []
with open("../results/sampled_individuals_low.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        genotypes.append(obj)

with open("../configs/klejda_gae_config.yaml") as f:
    config = yaml.safe_load(f)

dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])
dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    persistent_workers=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

## GAE

In [11]:
modelGAE = GraphAutoencoder(config)
wandb_logger = WandbLogger(project="Framsticks-GAE", name="GAE-Baseline-Test", save_dir = config["save_dir"])

trainer = pl.Trainer(
    max_epochs=10,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)
trainer.fit(modelGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │  111 K │ train │     0 │
│ 1 │ fc_z             │ Linear   │  1.0 K │ train │     0 │
│ 2 │ decoder_a        │ DecoderA │ 90.4 K │ train │     0 │
│ 3 │ decoder_x        │ DecoderX │ 52.0 K │ train │     0 │
│ 4 │ criterion        │ MSELoss  │      0 │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 255 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 255 K                                                                                                
Total estimated model params size (MB): 1.021                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=10` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 18-19, summary
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
wandb: train/locality_correlation ▁▅▆▇▇████▄
wandb:               train/loss_A █▂▂▂▂▂▂▂▂▁
wandb:               train/loss_X █▇▇▇▇▇▆▂▂▁
wandb:        train/loss_locality █▄▃▂▂▁▁▁▁▅
wandb:           train/loss_total █▂▂▂▂▂▂▂▂▁
wandb:        trainer/global_step ▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
wandb:   val/locality_correlation ▁▃▆▆▆▇█▂█▃
wandb:                 val/loss_A █████████▁
wandb:                 val/loss_X ██████▄▄▄▁
wandb:                         +2 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 9
wandb: train/locality_correlation 1.8655
wandb:               train/loss_A 0.00205
wandb:               train/loss_X 0.09977
wandb:        train/loss_locality -0.8655
wandb:           train/loss_total 1.71446
wandb:        trainer/global_step 10499
wandb:   val/loca

## VGAE

In [ ]:
modelVGAE = VariationalGraphAutoencoder(config)

wandb_logger = WandbLogger(project="Framsticks-VGAE", name="VGAE-Baseline-Test", save_dir = config["save_dir"])
trainer = pl.Trainer(
    max_epochs=10,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)

trainer.fit(modelVGAE, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
wandb.finish()